# Hands-on 2: Applying large models to forecasting and QA

**ACML 2026 Tutorial** · *Advances in Multi-Modal Time Series: From Forecasting to Reasoning*
Van Dai Do, Hiep Nguyen, Hung Le · A²I², Deakin University

Hands-on 1 trained a paired time-series and text model from scratch; this notebook uses models that are already pretrained. First: **Runtime › Change runtime type › T4 GPU** — every section fits on a free T4. After Section 1 you can skip around: Section 5 reuses Chronos-2 from Section 2, Section 6 reuses the questions from Section 3, and Sections 5–7 are take-homes.

| § | What you do | Models | Time on a T4 |
|---|---|---|---|
| 1 | Load nine Time-MMD domains | – | < 1 min |
| 2 | Forecast the nine domains zero-shot | Chronos-2, TimesFM 2.5 | ~2 min |
| 3 | Ask an LLM about time series; run the shuffle test | Qwen3-1.7B | ~3 min |
| 4 | Run Lotus, a forecast-grounded TSLM | Lotus (Qwen3-0.6B) | ~6 min |
| 5–7 | Take-homes: text as covariate · chart as input · train a tiny TSLM | Chronos-2 · Qwen3-VL-2B · Qwen3-0.6B | ~7 min |

In [ ]:
%%capture
!pip install -q "chronos-forecasting==2.3.2" "timesfm[torch]==3.0.2" "transformers>=4.57,<6" accelerate datasets peft

In [ ]:
import gc, re, random, textwrap
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    # bfloat16 on Ampere or newer GPUs. A T4 has no bfloat16, and Qwen models can overflow in float16,
    # so we keep float32 there: every model in this notebook still fits in the T4's 15 GB.
    LLM_DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float32
    print("GPU:", torch.cuda.get_device_name(0), "| LLM dtype:", LLM_DTYPE)
else:
    LLM_DTYPE = torch.float32
    print("No GPU found. The notebook still runs, but slowly: Runtime > Change runtime type > T4 GPU.")

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 1. Data: Time-MMD

[Time-MMD](https://github.com/AdityaLab/Time-MMD) (Liu et al., NeurIPS 2024) pairs a numerical series with text reports in ten domains; we use the target column `OT` of nine of them (`Health_AFR` is left out — its target has missing and infinite values).

*Cells: `load_domain` reads and cleans one domain CSV; the next cell loads the nine domains and plots their last 300 steps.*

In [ ]:
# read one Time-MMD csv: clean OT, parse dates, infer D/W/M frequency
def load_domain(name):
    df = pd.read_csv(f"Time-MMD/numerical/{name}/{name}.csv")
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").drop_duplicates("date")
    y = pd.to_numeric(df["OT"], errors="coerce").replace([np.inf, -np.inf], np.nan)
    y = y.interpolate().bfill().ffill()
    step = df["date"].diff().dt.days.median()
    freq = "D" if step <= 1 else ("W" if step <= 8 else "M")
    return pd.Series(y.values, index=df["date"].values, name=name), freq

In [ ]:
!git clone -q --depth 1 https://github.com/AdityaLab/Time-MMD.git 2>/dev/null || echo "Time-MMD already cloned"

DOMAINS = ["Agriculture", "Climate", "Economy", "Energy", "Environment",
           "Health_US", "Security", "SocialGood", "Traffic"]
SEASON = {"D": 7, "W": 52, "M": 12}          # seasonal period used by the seasonal-naive baseline

series = {d: load_domain(d) for d in DOMAINS}
for d, (y, f) in series.items():
    print(f"{d:12s} freq={f}  length={len(y):6d}  {y.index[0]:%Y-%m-%d} to {y.index[-1]:%Y-%m-%d}")

fig, axes = plt.subplots(3, 3, figsize=(15, 7))
for ax, (d, (y, f)) in zip(axes.flat, series.items()):
    tail = y.iloc[-300:]
    ax.plot(tail.index, tail.values, lw=1.2)
    ax.set_title(f"{d} ({f})", fontsize=10)
    ax.tick_params(labelsize=8)
plt.tight_layout(); plt.show()

## 2. Zero-shot forecasting with numerical foundation models

Two models from Part 4, used **zero-shot** — neither was trained on Time-MMD. **Chronos-2** (Amazon, 120M): a patch encoder that also accepts multivariate targets and covariates. **TimesFM 2.5** (Google, 200M): a decoder-only transformer over 32-step input patches with a quantile head.

We forecast the last four windows of each domain (horizon 8) and report MAE **relative to the naive forecast** that repeats the last value — below 1.0 means better than naive.

*Cells: load — both pipelines; functions — windowing, the two baselines, the two forecast wrappers; run — four windows per domain, MAE table, three example forecasts.*

In [ ]:
from chronos import Chronos2Pipeline
import timesfm

H, N_WINDOWS, MAX_CTX = 8, 4, 1024

chronos2 = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map=DEVICE)

tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch", torch_compile=False)
tfm.compile(timesfm.ForecastConfig(
    max_context=MAX_CTX, max_horizon=128, normalize_inputs=True, per_core_batch_size=8,
    use_continuous_quantile_head=True, force_flip_invariance=True,
    infer_is_positive=True, fix_quantile_crossing=True))
print("Both models loaded.")

In [ ]:
def windows(y, h=H, n=N_WINDOWS):
    # Last n non-overlapping (context, target) windows of a series.
    v = y.values.astype("float32")
    for k in range(n, 0, -1):
        end = len(v) - k * h
        yield v[max(0, end - MAX_CTX):end], v[end:end + h]

# baseline: repeat the last observed value
def naive(ctx, h, m):
    return np.repeat(ctx[-1], h)

# baseline: repeat the value from one season ago
def seasonal_naive(ctx, h, m):
    if len(ctx) < m:
        return naive(ctx, h, m)
    return np.array([ctx[-m + (i % m)] for i in range(h)])

# Chronos-2: median + 10/50/90% quantiles (first variate)
def chronos2_forecast(ctxs, h):
    q, med = chronos2.predict_quantiles([torch.tensor(c) for c in ctxs], prediction_length=h,
                                        quantile_levels=[0.1, 0.5, 0.9])
    # One tensor per series, shaped (n_variates, h, n_quantiles) and (n_variates, h).
    return (np.stack([m[0].float().cpu().numpy() for m in med]),
            np.stack([x[0].float().cpu().numpy() for x in q]))

# TimesFM: point + 10/50/90% deciles (channel 0 is the mean)
def timesfm_forecast(ctxs, h):
    point, quant = tfm.forecast(horizon=h, inputs=[c.copy() for c in ctxs])
    return point, quant[..., [1, 5, 9]]      # channel 0 is the mean, channels 1 to 9 are deciles 0.1 to 0.9

In [ ]:
rows, examples = [], {}
for d, (y, f) in series.items():
    m = SEASON[f]
    ctxs, tgts = map(list, zip(*windows(y)))
    tgt = np.stack(tgts)
    preds = {"Naive": np.stack([naive(c, H, m) for c in ctxs]),
             "Seasonal naive": np.stack([seasonal_naive(c, H, m) for c in ctxs])}
    preds["Chronos-2"], q_c2 = chronos2_forecast(ctxs, H)
    preds["TimesFM 2.5"], q_tf = timesfm_forecast(ctxs, H)
    mae = {k: np.abs(v - tgt).mean() for k, v in preds.items()}
    rows.append({"domain": d, **{k: mae[k] / mae["Naive"] for k in preds}})
    examples[d] = (ctxs[-1], tgts[-1], q_c2[-1], q_tf[-1])

table = pd.DataFrame(rows).set_index("domain")
table.loc["Average"] = table.mean()
display(table.round(3).style.highlight_min(axis=1, color="#d9f2d9").format("{:.3f}"))

fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
for ax, d in zip(axes, ["Health_US", "Energy", "Traffic"]):
    ctx, tgt, qc, qt = examples[d]
    n = 6 * H
    ax.plot(np.arange(-n, 0), ctx[-n:], color="0.35", lw=1.5, label="history")
    ax.plot(np.arange(H), tgt, color="black", lw=2, label="truth")
    for name, q, c in [("Chronos-2", qc, "tab:blue"), ("TimesFM 2.5", qt, "tab:orange")]:
        ax.plot(np.arange(H), q[:, 1], color=c, lw=2, label=f"{name} median")
        ax.fill_between(np.arange(H), q[:, 0], q[:, 2], color=c, alpha=0.2)
    ax.axvline(-0.5, color="0.7", ls=":")
    ax.set_title(d)
axes[0].legend(fontsize=8, loc="upper left")
plt.tight_layout(); plt.show()

**Discuss.** Where do the foundation models beat seasonal-naive, and where not? Compare with how seasonal each domain looks in Section 1. The shaded bands are 80% intervals — both models give you uncertainty for free.

**Optional: TimesFM 3.0.** Released August 2026, it adds native multivariate forecasting and covariates, but its weights carry a non-commercial licence (the table above uses the Apache-licensed 2.5). If the weights are gated, accept the licence on the model page and run `from huggingface_hub import login; login()` first — or skip this cell.

In [ ]:
from timesfm3 import TimesFM3Forecaster

tfm3 = None
try:
    tfm3 = TimesFM3Forecaster.from_pretrained("google/timesfm-3.0-pytorch", device=DEVICE)
except Exception as e:                                 # gated weights or no network: skip, nothing below depends on it
    print(f"TimesFM 3.0 weights not available ({type(e).__name__}). Accept the licence at "
          "https://huggingface.co/google/timesfm-3.0-pytorch, run `from huggingface_hub import login; login()`, and rerun.")

if tfm3 is not None:
    for d, (y, f) in series.items():
        ctxs, tgts = map(list, zip(*windows(y)))
        outs = list(tfm3.predict_batch(contexts=[c.astype("float32") for c in ctxs], horizon=H))
        point = np.stack([o.forecast for o in outs])
        naive_mae = np.abs(np.stack([naive(c, H, SEASON[f]) for c in ctxs]) - np.stack(tgts)).mean()
        table.loc[d, "TimesFM 3.0"] = np.abs(point - np.stack(tgts)).mean() / naive_mae
    table.loc["Average", "TimesFM 3.0"] = table.loc[DOMAINS, "TimesFM 3.0"].mean()
    display(table.round(3))

## 3. Question answering with a language model, and the shuffle test

Time-MQA (Part 3) writes the values of a series into the prompt and asks a question. We do the same with an off-the-shelf instruction model, **Qwen3-1.7B**, with no time-series training at all. Questions are scored exactly: 48-step windows from Section 1, scaled to 0–100, in Time-MQA's true-or-false and multiple-choice formats.

| Type | Question | Chance |
|---|---|---|
| `trend` | Is the last quarter higher on average than the first quarter? (T/F) | 50% |
| `peak` | Which quarter holds the maximum? (A–D) | 25% |
| `volatility` | Which half varies more? (A/B) | 50% |
| `spike` | Is there one isolated spike far above the rest? (T/F) | 50% |

**The shuffle test.** Every question is asked again with the same values in random order. Shuffling destroys the answer to `trend`, `peak` and `volatility`, so a model that reads the numbers should fall to chance on them; it keeps the answer to `spike` (a spike is still there after shuffling). Time-MQA ran a related check and found that swapping in a random series did not lower accuracy.

*Cells: functions — question sampler, answer parser, scorer, prompt builder, asker; build — sample the 48 questions; load — Qwen3-1.7B and one demo answer; run — original vs shuffled accuracy.*

In [ ]:
WIN = 48   # question window length, in steps

# one random 48-step window of a domain, rescaled to 0-100
def scaled_window(y, rng, win=WIN):
    v = y.values
    start = int(rng.integers(0, len(v) - win))
    w = v[start:start + win].astype(float)
    lo, hi = w.min(), w.max()
    return np.round(100 * (w - lo) / (hi - lo + 1e-9), 1)

# build one question of a given kind, with a computable answer
def make_item(kind, rng):
    while True:
        d = DOMAINS[int(rng.integers(len(DOMAINS)))]
        w = scaled_window(series[d][0], rng)
        if w.std() > 5:                       # skip flat windows
            break
    q = WIN // 4
    if kind == "trend":
        gap = w[-q:].mean() - w[:q].mean()
        if abs(gap) < 5:                      # avoid near ties
            return make_item(kind, rng)
        return dict(kind=kind, values=w, answer="True" if gap > 0 else "False", options=None,
                    question="True or false: the values in the last quarter are higher on average than in the first quarter.")
    if kind == "peak":
        return dict(kind=kind, values=w, answer="ABCD"[int(np.argmax(w)) // q],
                    options="A) first quarter  B) second quarter  C) third quarter  D) fourth quarter",
                    question="In which quarter of the series is the maximum value?")
    if kind == "volatility":
        s1, s2 = np.std(np.diff(w[:WIN // 2])), np.std(np.diff(w[WIN // 2:]))
        if abs(s1 - s2) < 0.25 * max(s1, s2):
            return make_item(kind, rng)
        return dict(kind=kind, values=w, answer="A" if s1 > s2 else "B",
                    options="A) the first half  B) the second half",
                    question="Which half of the series fluctuates more from step to step?")
    if kind == "spike":
        has = bool(rng.integers(2))
        w = w.copy()
        if has:                               # squeeze the window, then add one isolated spike
            w = np.round(w * 0.6, 1)
            w[int(rng.integers(4, WIN - 4))] = 100.0
        elif np.sort(w)[-2] < 80:             # a "no spike" window must not have an isolated maximum
            return make_item(kind, rng)
        return dict(kind=kind, values=w, answer="True" if has else "False", options=None,
                    question="True or false: the series contains one isolated spike far above all other values.")

# pull True/False or the option letter out of a model answer
def parse(item, out):
    if item["options"] is None:
        m = re.search(r"\b(true|false)\b", out.lower())
        return m.group(1).capitalize() if m else None
    m = re.search(r"\b([ABCD])\b", out.upper())
    return m.group(1) if m else None

# accuracy and parse rate per question kind
def accuracy_by_kind(items, outputs):
    rows = [{"kind": it["kind"], "correct": parse(it, out) == it["answer"], "parsed": parse(it, out) is not None}
            for it, out in zip(items, outputs)]
    return pd.DataFrame(rows).groupby("kind", sort=False).agg(accuracy=("correct", "mean"), parsed=("parsed", "mean"))

# render an item as the text prompt for the LM
def prompt_for(item, values):
    fmt = "Answer with True or False only." if item["options"] is None else f"Options: {item['options']}\nAnswer with the letter only."
    series_txt = ", ".join(f"{x:g}" for x in values)
    return (f"Here is a time series of {len(values)} values, scaled to 0 to 100:\n{series_txt}\n\n"
            f"{item['question']}\n{fmt}")

# greedy answer to one prompt
@torch.no_grad()
def ask(prompts, max_new_tokens=8):
    outs = []
    for p in prompts:
        text = qa_tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False,
                                          add_generation_prompt=True, enable_thinking=False)
        enc = qa_tok(text, return_tensors="pt").to(DEVICE)
        gen = qa_lm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=qa_tok.eos_token_id)
        outs.append(qa_tok.decode(gen[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip())
    return outs

In [ ]:
Q_RNG = np.random.default_rng(42)

KINDS = ["trend", "peak", "volatility", "spike"]
items = [make_item(k, Q_RNG) for k in KINDS for _ in range(12)]

print(len(items), "questions. Example:")
print(items[1]["question"], items[1]["options"], "| answer:", items[1]["answer"])

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

QA_MODEL = "Qwen/Qwen3-1.7B"
qa_tok = AutoTokenizer.from_pretrained(QA_MODEL)
qa_lm = AutoModelForCausalLM.from_pretrained(QA_MODEL, dtype=LLM_DTYPE, device_map=DEVICE).eval()

print(prompt_for(items[1], items[1]["values"]))
print("\nModel says:", ask([prompt_for(items[1], items[1]["values"])])[0])

In [ ]:
shuffle_rng = np.random.default_rng(7)
orig_out = ask([prompt_for(it, it["values"]) for it in items])
shuf_out = ask([prompt_for(it, shuffle_rng.permutation(it["values"])) for it in items])

res = accuracy_by_kind(items, orig_out).rename(columns={"accuracy": "original", "parsed": "parsed (orig.)"})
res["shuffled"] = accuracy_by_kind(items, shuf_out)["accuracy"]
res["chance"] = [0.5, 0.25, 0.5, 0.5]
res.round(2)

**Read the table.** If `original` and `shuffled` are close for `trend`, `peak` and `volatility`, the model is answering from the question and its priors, not from the series. The `parsed` column says how often the answer format was followed at all — in the Lotus paper, zero-shot 7–8B models scored below chance on sensor tasks, and missed formats are one likely reason. *(The real TSQA data is gated: accept the terms at `huggingface.co/datasets/Time-MQA/TSQA` to try it.)*

## 4. Lotus: a forecast-grounded time-series language model

Lotus (Part 4) maps temporal-spectral patches into a small language model and is pretrained so that its series representations must keep predicting the next patch — **forecast grounding** — before it is adapted to downstream tasks. The released checkpoint we load is that Phase-1 (pretrained) model: a frozen Qwen3-0.6B backbone — the same model Section 7 fine-tunes — plus a 3M-parameter patch encoder, a 2M-parameter next-patch decoder and LoRA adapters, about 10M trained parameters in total.

We run it three ways, each in the exact prompt format of one of its pretraining corpora:

* **Question answering** (TSQA): label the trend of a series as constant, upward or downward.
* **Captioning** (M4): describe a series in words.
* **Forecasting** (Chronos): the hidden states at the series tokens must predict the next 32-step patch — the grounding objective itself.
* **Downstream-adapted (Phase 2)**: the checkpoint after joint fine-tuning on HAR, Sleep-EDF, ECG-QA, PAMAP2 and TimeMQA, evaluated on real accelerometer windows from its HAR test split — read its rationale, then its accuracy.

*Cells: load — Lotus Phase-1 from its released checkpoint; functions — the TSQA asker, the M4 captioner, the next-patch forecaster; language — trend QA and one caption; forecast — two autoregressive patches vs naive; phase-2 — swap in the five-task fine-tuned checkpoint and evaluate it on 64 real HAR test windows.*

In [ ]:
if "qa_lm" in globals():                          # Section 3 ran: free its GPU memory
    del qa_lm; free_gpu()
!git clone -q --depth 1 https://github.com/DA2I2-SLM/Lotus.git 2>/dev/null || echo "Lotus already cloned"
!pip install -q peft
!pip uninstall -q -y torchao   # Colab's preinstalled torchao is older than peft allows; Lotus never uses it

import sys
from types import SimpleNamespace
sys.path.insert(0, "Lotus/src")

from huggingface_hub import hf_hub_download
from opentslm.model.llm.OpenTSLMSPFixPos2D import OpenTSLMSPFixPos2D
from opentslm.time_series_datasets.util import extend_time_series_to_match_patch_size_and_aggregate

HF_REPO = "hiepnh137/LangTimeFoundationModel"
CKPT_FILE = "qwen3_0_6b_joint_phase1_extend_balanced_lora_r8_freq_mvpos/checkpoints/best_model.pt"

# Architecture of the released run (configs/qwen3_0_6b.env in the Lotus repo): transformer
# encoder and decoder of width 192, frequency branch on, LoRA r=8 alpha=16 on the frozen LLM.
lotus_cfg = SimpleNamespace(
    is_pretraining=True, patch_size=32, pe_type="sincos",
    use_only_time_pe=False, use_only_channel_pe=False,
    use_freq_embed=True, enc_dec_type="transformer",
    trans_enc_dim=192, trans_enc_num_layers=6, trans_enc_num_heads=8, trans_enc_ff_dim=896,
    trans_dec_dim=192, trans_dec_num_layers=4, trans_dec_num_heads=8, trans_dec_ff_dim=896,
    llm_mv_position_ids=True,
    llm_dtype="bf16" if LLM_DTYPE == torch.bfloat16 else "fp32",
)

lotus = OpenTSLMSPFixPos2D(lotus_cfg, llm_id="Qwen/Qwen3-0.6B", device=DEVICE)
lotus.enable_lora(lora_r=8, lora_alpha=16)           # the checkpoint carries LoRA adapters
lotus.load_from_file(hf_hub_download(HF_REPO, CKPT_FILE))
lotus.eval()
print("Lotus loaded: encoder, projector, next-patch decoder and LoRA adapters restored.")

In [ ]:
import torch.nn.functional as F

# trend question in Lotus's TSQA pretraining format
def ask_lotus(question, values, max_new_tokens=8):
    m, s = float(np.mean(values)), float(np.std(values))
    sample = {                                        # Phase-1 TSQA prompt (Phase1JointDataset)
        "pre_prompt": question,
        "time_series": [list(map(float, values))],    # raw values; the model z-normalises
        "time_series_text": [f"This is a univariate time series with 1 channel and {len(values)} "
                             f"time steps. It has mean {m:.4f} and std {s:.4f}."],
        "post_prompt": "Predict the Trend Answer:",
    }
    batch = extend_time_series_to_match_patch_size_and_aggregate([sample], patch_size=32, normalize=True)
    text, _ = lotus.generate(batch, max_new_tokens=max_new_tokens)
    return text[0].strip()

# caption request in Lotus's M4 pretraining format
def caption_lotus(values, max_new_tokens=64):
    m, s = float(np.mean(values)), float(np.std(values))
    sample = {                                        # Phase-1 M4 captioning prompt
        "pre_prompt": "You are an expert in time series analysis.",
        "time_series": [list(map(float, values))],
        "time_series_text": [f"This is a univariate time series with 1 channel and {len(values)} "
                             f"time steps. It has mean {m:.4f} and std {s:.4f}:"],
        "post_prompt": "Please generate a detailed caption for this time-series, "
                       "describing it as accurately as possible.",
    }
    batch = extend_time_series_to_match_patch_size_and_aggregate([sample], patch_size=32, normalize=True)
    text, _ = lotus.generate(batch, max_new_tokens=max_new_tokens)
    return text[0].strip()

@torch.no_grad()
def lotus_forecast(values, n_patches):
    # The forecast-grounding read-out of pretraining: hidden states at the series tokens go
    # through decoder_projector and the next-patch decoder (compute_pretrain_loss).
    hist = np.asarray(values, dtype="float32")
    mean, std = float(hist.mean()), max(float(hist.std()), 1e-8)
    hist_n, hist_len = ((hist - mean) / std).tolist(), len(hist)
    out = []
    for _ in range(n_patches):
        sample = [{                                   # Phase-1 forecasting prompt
            "pre_prompt": "",
            "time_series": [hist_n],
            "time_series_text": [f"This is a univariate time series with 1 channel and {hist_len} "
                                 f"time steps. It has mean {mean:.4f} and std {std:.4f}. <ts>"],
            "post_prompt": lotus.get_eos_token(),
            "norm_mean": mean, "norm_std": std,
            "time_series_norm_stats": [(mean, std)],
            "ts_already_normalized": True,
        }]
        batch = extend_time_series_to_match_patch_size_and_aggregate(sample, patch_size=32)
        embeds, attn, _, ts_mask, _, _, _ = lotus.pad_and_apply_batch(batch, return_targets=True)
        hidden = lotus.llm(inputs_embeds=embeds, attention_mask=attn,
                           output_hidden_states=True, return_dict=True).hidden_states[-1]
        h = lotus.decoder_projector(hidden[ts_mask].float())
        h = F.layer_norm(h, [h.size(-1)])
        patch = lotus.decoder(h[None])[:, -1].float().cpu().numpy()[0]  # last patch state -> next patch
        out.append(patch)
        hist_n = (hist_n + patch.tolist())[-hist_len:]                  # slide the window by one patch
    return np.concatenate(out) * std + mean

In [ ]:
TREND_Q = ("Given the following definitions:\n"
           "Constant trend: The time series does not show any significant increase or decrease over time.\n"
           "Upward trend: The time series consistently increases over time.\n"
           "Downward trend: The time series consistently decreases over time.\n"
           "Select one of the following answers that best describes the provided time series:\n"
           "(a) This time series has a constant trend.\n"
           "(b) This time series has an upward trend.\n"
           "(c) This time series has a downward trend.\n"
           "Only answer (a), (b), or (c).")

rng = np.random.default_rng(4)
t = np.arange(128)
cases = {"(b)": (2.0 * t / 128 + rng.normal(0, 0.1, 128)).astype("float32"),
         "(c)": (2.0 - 2.0 * t / 128 + rng.normal(0, 0.1, 128)).astype("float32"),
         "(a)": (1.0 + rng.normal(0, 0.1, 128)).astype("float32")}
for truth, v in cases.items():
    print(f"truth {truth}  ->  Lotus says {ask_lotus(TREND_Q, v)}")
print(f"same series shuffled ->  Lotus says {ask_lotus(TREND_Q, rng.permutation(cases['(b)']))}"
      "  (shuffling destroys the trend, so (a) is now correct)")

rng = np.random.default_rng(5)
t = np.arange(256)                                    # rising trend with a 64-step cycle
w = (3.0 * t / 256 + np.sin(2 * np.pi * t / 64) + rng.normal(0, 0.15, 256)).astype("float32")
print(textwrap.fill(caption_lotus(w), 100))

In [ ]:
if "series" in globals():                              # Section 1 ran: use a real domain
    y = series["Traffic"][0].values[-320:].astype("float32"); name = "Traffic (Time-MMD)"
else:
    rng = np.random.default_rng(6)
    t = np.arange(320)
    y = (2.0 * t / 320 + np.sin(2 * np.pi * t / 64) + rng.normal(0, 0.15, 320)).astype("float32")
    name = "synthetic (rising trend, 64-step cycle)"
ctx, tgt = y[:-64], y[-64:]
pred = lotus_forecast(ctx, n_patches=2)
naive = np.repeat(ctx[-1], 64)
print(f"{name}: MAE over the next 64 steps - Lotus {np.abs(pred - tgt).mean():.3f}, "
      f"naive {np.abs(naive - tgt).mean():.3f}")

fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(np.arange(-96, 0), ctx[-96:], color="0.35", lw=1.5, label="history")
ax.plot(np.arange(64), tgt, color="black", lw=2, label="truth")
ax.plot(np.arange(64), pred, color="tab:green", lw=2, label="Lotus (next 2 patches)")
ax.plot(np.arange(64), naive, color="0.7", ls="--", label="naive")
ax.axvline(-0.5, color="0.7", ls=":")
ax.set_title(name); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
import ast

# Phase-2 checkpoint, and a balanced 64-window sample of its HAR test split
P2_REPO, P2_CKPT = "daidv1112/FoundationalTS", "lotus_qwen3-0.6b_joint-5task-finetune.pt"
HAR_SUBSET_URL = ("https://raw.githubusercontent.com/Davido111200/lotus-hands-on-2-fixed/"
                  "main/har_test_subset64.csv")
LABELS = ["biking", "lying", "running", "sitting", "standing", "walking", "walking_down", "walking_up"]

HAR_PROMPT = """
        You are given accelerometer data in all three dimensions. Your task is to classify the activity based on analysis of the data.

        Instructions:
        - Begin by analyzing the time series without assuming a specific label.
        - Think step-by-step about what the observed patterns suggest regarding movement intensity and behavior.
        - Write your rationale as a single, natural paragraph — do not use bullet points, numbered steps, or section headings.
        - Do **not** mention any class label until the final sentence.

        Possible activity labels are:
        biking, lying, running, sitting, standing, walking, walking_down, walking_up.

        - Make sure that your last word is the answer. You MUST end your response with "Answer: "
        """

# one real accelerometer window in the exact HAR Phase-2 prompt format
def har_sample(row):
    series = torch.tensor([row["x_axis"], row["y_axis"], row["z_axis"]], dtype=torch.float32)
    means, stds = series.mean(dim=1, keepdim=True), series.std(dim=1, keepdim=True).clamp(min=1e-6)
    texts = [f"The following is the accelerometer data on the {ax}-axis, it has mean {m:.4f} and std {s:.4f}:"
             for ax, m, s in zip("xyz", means.flatten().tolist(), stds.flatten().tolist())]
    return {"pre_prompt": HAR_PROMPT, "time_series": ((series - means) / stds).tolist(),
            "time_series_text": texts, "post_prompt": "Rationale:", "answer": ""}

# the activity named after the final "Answer:"
def parse_har(out):
    hits = re.findall(r"Answer:\s*\**\s*([A-Za-z_]+)", out)
    if not hits:
        return None
    word = hits[-1].lower().strip(".,")
    return next((lab for lab in LABELS if lab in word or word in lab), None)

In [ ]:
# Phase 2: the same architecture after joint adaptation on HAR, Sleep-EDF, ECG-QA,
# PAMAP2 and TimeMQA. The new weights replace the Phase-1 ones inside `lotus`.
lotus.load_from_file(hf_hub_download(P2_REPO, P2_CKPT))

har = pd.read_csv(HAR_SUBSET_URL)
for c in ("x_axis", "y_axis", "z_axis"):
    har[c] = har[c].map(ast.literal_eval)

correct, parsed, outputs = 0, 0, []
for _, row in har.iterrows():
    batch = extend_time_series_to_match_patch_size_and_aggregate([har_sample(row)], patch_size=32)
    text, _ = lotus.generate(batch, max_new_tokens=170)
    pred = parse_har(text[0])
    outputs.append((row["label"], text[0].strip()))
    parsed += pred is not None
    correct += pred == row["label"]
print(f"HAR test sample: {correct}/{len(har)} correct ({correct / len(har):.0%}), "
      f"answer parsed {parsed}/{len(har)}  (paper, full split: 63.7%)\n")
print("One rationale, verbatim:\n")
print(f"[truth: {outputs[0][0]}]\n{outputs[0][1]}")

**Discuss.** The trend answers track the series, and shuffling the values changes the answer: the numbers reach the model through the encoder's soft tokens, not through text. The forecast comes from the same representations the language head reads — forecast grounding, asking the language-aligned representation to stay predictive of the future. The first three demos run the Phase-1 checkpoint; the last swaps in the Phase-2 (five-task) weights — same architecture, same soft-token interface, only the training changed. The paper's headline numbers are Phase-2, and the repo's launchers reproduce them. The same Hugging Face repo also holds the Gemma3-270M/1B runs (licence-gated) — swap `CKPT_FILE`, `llm_id` and the `lotus_cfg` values from the repo README to try them.

## 5. Take-home: turning report text into a covariate

The Energy domain comes with weekly reports on gasoline prices, each holding an outlook sentence such as *"In the short term (next 1-3 months), gasoline prices are likely to decrease…"*. Chronos-2 accepts **categorical covariates**, so the simplest way to let it see the text is to reduce each outlook to a label (`up` / `down` / `stable` / `mixed`) and pass it as a past covariate.

One detail matters: a report filed in the week starting on Monday *d* describes the price published on Monday *d + 7 days*, so we attach it to *d + 7* — attaching it to *d* would leak a future price into the past. Between reports the last label is carried forward.

*Cells: `outlook_label` maps a report's outlook sentence to up/down/stable/mixed; the next cell attaches the labels to weeks and compares Chronos-2 with and without them.*

In [ ]:
def outlook_label(preds):
    # preds holds "long-term outlook;short-term outlook"; we read the short-term part.
    parts = str(preds).split(";")
    text = (parts[1] if len(parts) > 1 else parts[0]).lower()
    up = re.search(r"\b(increas|rise|rising|higher|climb|upward)", text) is not None
    down = re.search(r"\b(decreas|declin|fall|lower|drop|downward)", text) is not None
    if up != down:
        return "up" if up else "down"
    stable = re.search(r"\b(stable|stabili|steady|unchanged)", text) is not None
    return "stable" if stable and not up else "mixed"

In [ ]:
if "lotus" in globals():                        # Section 4 ran: free its GPU memory
    del lotus; free_gpu()
if "chronos2" not in globals():                       # Section 2 skipped: load the model here
    from chronos import Chronos2Pipeline
    chronos2 = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map=DEVICE)

H = 8                                                  # forecast horizon, as in Section 2
energy, _ = series["Energy"]
rep = pd.read_csv("Time-MMD/textual/Energy/Energy_report.csv", parse_dates=["start_date"])
rep["date"] = rep["start_date"] + pd.Timedelta(days=7)

rep["outlook"] = rep["preds"].map(outlook_label)
print(rep["outlook"].value_counts().to_string(), "\n")
for _, r in rep.sample(3, random_state=1).iterrows():
    print(f"{r['date']:%Y-%m-%d}  [{r['outlook']}]  {textwrap.shorten(str(r['preds']).split(';')[-1], 150)}")

# Align labels to the weekly price index, carrying the last report forward.
labels = (rep.sort_values("date").drop_duplicates("date", keep="last").set_index("date")["outlook"]
             .reindex(energy.index, method="ffill").fillna("mixed"))
first_report = rep["date"].min()
y_sub = energy[energy.index >= first_report]
lab_sub = labels[labels.index >= first_report].values.astype(str)

errs = {"Chronos-2": [], "Chronos-2 + outlook": []}
n_win = 12
for k in range(n_win, 0, -1):
    end = len(y_sub) - k * H
    ctx, tgt = y_sub.values[:end].astype("float32"), y_sub.values[end:end + H]
    _, plain = chronos2.predict_quantiles([torch.tensor(ctx)], prediction_length=H, quantile_levels=[0.5])
    _, with_text = chronos2.predict_quantiles(
        [{"target": ctx, "past_covariates": {"outlook": lab_sub[:end]}}],
        prediction_length=H, quantile_levels=[0.5])
    errs["Chronos-2"].append(np.abs(plain[0][0].float().cpu().numpy() - tgt).mean())
    errs["Chronos-2 + outlook"].append(np.abs(with_text[0][0].float().cpu().numpy() - tgt).mean())
print({k: round(float(np.mean(v)), 4) for k, v in errs.items()})

**Discuss.** A single keyword label is a crude summary of a paragraph, and the outlook may repeat what the prices already show. Parts 3 and 4 are about models that read the text itself instead of a hand-made feature.

## 6. Take-home: show the plot instead of the numbers

Part 3's second route: render the series as a chart and ask a vision-language model. We reuse the questions from Section 3 with **Qwen3-VL-2B-Instruct**, simply drawing a line chart — as Merrill et al., and Zhou and Yu (ICLR 2025), did when they gave plots to vision-language models. Time-VLM learns its own rendering for forecasting; here the chart itself is the interface.

*Cells: functions — chart renderer and `ask_vlm`; run — Qwen3-VL-2B answers the Section-3 questions from images.*

In [ ]:
# draw a 48-step series as a small line chart
def render(values):
    fig, ax = plt.subplots(figsize=(4.48, 2.24), dpi=100)
    ax.plot(values, color="black", lw=1.5)
    ax.set_xticks(np.linspace(0, len(values), 5)); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.canvas.draw()
    img = Image.fromarray(np.asarray(fig.canvas.buffer_rgba())[..., :3].copy())
    plt.close(fig)
    return img

# greedy answer to one chart + question
@torch.no_grad()
def ask_vlm(item, values, max_new_tokens=8):
    fmt = "Answer with True or False only." if item["options"] is None else f"Options: {item['options']}\nAnswer with the letter only."
    messages = [{"role": "user", "content": [
        {"type": "image", "image": render(values)},
        {"type": "text", "text": f"The chart shows a time series of {len(values)} steps.\n{item['question']}\n{fmt}"}]}]
    inputs = proc.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                      return_dict=True, return_tensors="pt").to(DEVICE)
    out = vlm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return proc.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()

In [ ]:
from PIL import Image
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

VLM_NAME = "Qwen/Qwen3-VL-2B-Instruct"
vlm = Qwen3VLForConditionalGeneration.from_pretrained(VLM_NAME, dtype=LLM_DTYPE, device_map=DEVICE).eval()
proc = AutoProcessor.from_pretrained(VLM_NAME)

display(render(items[1]["values"]))
vlm_out = [ask_vlm(it, it["values"]) for it in items]
vlm_res = accuracy_by_kind(items, vlm_out).rename(columns={"accuracy": "image (VLM)", "parsed": "parsed (VLM)"})
if "res" in globals():                                   # Section 3 ran: put the columns side by side
    vlm_res = res.join(vlm_res)
vlm_res.round(2)

## 7. Take-home: build a tiny forecast-grounded TSLM

This rebuilds the **forecast-grounding** recipe from Part 4 in miniature — not the Lotus architecture, only its core idea:

1. Cut each series into 32-step patches, each described by its raw values **and** its rFFT magnitudes (a temporal-spectral patch).
2. A small MLP maps every patch to one soft token in a frozen LM's embedding space; only LoRA adapters on its attention train, as in Lotus's first phase.
3. Two losses train the MLP, adapters and a small head: **language** (the LM writes a caption of the series) and **forecast grounding** (from the LM's hidden state at patch *i*, the head must predict patch *i + 1* — causal attention makes this a real forecasting task).

The data is synthetic — a trend (rising/flat/falling) plus a cycle of 8, 16, 32 or 64 steps plus noise, with the caption naming both — so the LM has to read the soft tokens to get it right. The backbone, Qwen3-0.6B, is one of the three used in the Lotus paper.

*Cells: functions — series generator, patching, encoder/head/LoRA modules, both losses, the evaluation helpers; setup — frozen Qwen3-0.6B with LoRA, encoder and head; train; eval; probe.*

In [ ]:
import torch.nn as nn, torch.nn.functional as F
P, N_PATCH = 32, 8     # 8 patches of 32 steps = 256 steps per series
PERIODS = [8, 16, 32, 64]
TRENDS = ["Falling", "Flat", "Rising"]

# synthetic series: trend + cycle + noise, with its caption
def make_series(rng, length=P * N_PATCH):
    t = np.arange(length)
    trend, period = int(rng.integers(3)), int(rng.choice(PERIODS))
    x = ((trend - 1) * rng.uniform(1.5, 3.0) * t / length
         + rng.uniform(0.5, 1.5) * np.sin(2 * np.pi * t / period + rng.uniform(0, 2 * np.pi))
         + rng.normal(0, 0.15, length))
    return x.astype("float32"), f"{TRENDS[trend]} trend with a cycle of {period} steps."

def to_patches(x):
    # Per-series standardisation, then (batch, N_PATCH, P).
    x = torch.as_tensor(np.asarray(x, dtype="float32"))
    x = x[None] if x.ndim == 1 else x
    mu, sd = x.mean(1, keepdim=True), x.std(1, keepdim=True) + 1e-5
    return ((x - mu) / sd).view(x.shape[0], N_PATCH, P)

class PatchEncoder(nn.Module):
    # Temporal-spectral patch -> one soft token in the LM embedding space.
    def __init__(self, d_model, hidden=512):
        super().__init__()
        n_in = P + P // 2 + 1
        self.norm = nn.LayerNorm(n_in)
        self.mlp = nn.Sequential(nn.Linear(n_in, hidden), nn.GELU(), nn.Linear(hidden, d_model))
        self.pos = nn.Parameter(torch.zeros(N_PATCH, d_model))
    def forward(self, patches):
        spectrum = torch.fft.rfft(patches, dim=-1).abs()
        return self.mlp(self.norm(torch.cat([patches, spectrum], dim=-1))) + self.pos

class NextPatchHead(nn.Module):
    # LM hidden state at patch i -> values of patch i + 1. Used only for training.
    def __init__(self, d_model, hidden=512):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(d_model, hidden), nn.GELU(), nn.Linear(hidden, P))
    def forward(self, h):
        return self.mlp(h)

class LoRALinear(nn.Module):
    # Frozen linear layer plus a trainable low-rank update B @ A (B starts at zero).
    def __init__(self, base, r=8, alpha=16):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.randn(r, base.in_features, device=base.weight.device) * 0.01)
        self.B = nn.Parameter(torch.zeros(base.out_features, r, device=base.weight.device))
    def forward(self, x):
        return self.base(x) + (x @ self.A.t() @ self.B.t()) * self.scale

# caption cross-entropy + next-patch MSE for a batch
def losses(patches, captions):
    B, n_prompt = patches.shape[0], len(PROMPT_IDS)
    cap_ids = [tok(" " + c, add_special_tokens=False).input_ids + [EOS] for c in captions]
    T = max(map(len, cap_ids))
    L = N_PATCH + n_prompt + T
    cap = torch.full((B, T), EOS, device=DEVICE)
    labels = torch.full((B, L), -100, device=DEVICE)
    mask = torch.zeros((B, L), dtype=torch.long, device=DEVICE)
    for i, c in enumerate(cap_ids):
        c = torch.tensor(c, device=DEVICE)
        cap[i, :len(c)] = c
        labels[i, N_PATCH + n_prompt:N_PATCH + n_prompt + len(c)] = c
        mask[i, :N_PATCH + n_prompt + len(c)] = 1
    prompt = embed(torch.tensor(PROMPT_IDS, device=DEVICE)).expand(B, -1, -1)
    inputs = torch.cat([encoder(patches), prompt, embed(cap)], dim=1)
    hidden = lm.model(inputs_embeds=inputs, attention_mask=mask).last_hidden_state
    logits = lm.lm_head(hidden).float()
    ce = F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)), labels[:, 1:].reshape(-1), ignore_index=-100)
    next_patch = head(hidden[:, :N_PATCH - 1].float())          # state at patch i predicts patch i + 1
    mse = F.mse_loss(next_patch, patches[:, 1:])
    return ce, mse

# generate a caption for one series
@torch.no_grad()
def describe(x, max_new_tokens=16):
    patches = to_patches(x).to(DEVICE)
    inputs = torch.cat([encoder(patches), embed(torch.tensor([PROMPT_IDS], device=DEVICE))], dim=1)
    out = lm.generate(inputs_embeds=inputs, attention_mask=torch.ones(inputs.shape[:2], dtype=torch.long, device=DEVICE),
                      max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=EOS)
    return tok.decode(out[0], skip_special_tokens=True).strip()

# next-patch MSE vs the repeat-current-patch baseline
@torch.no_grad()
def next_patch_mse(xs):
    patches = to_patches(np.stack(xs)).to(DEVICE)
    prompt = embed(torch.tensor(PROMPT_IDS, device=DEVICE)).expand(len(xs), -1, -1)
    hidden = lm.model(inputs_embeds=torch.cat([encoder(patches), prompt], dim=1)).last_hidden_state
    pred = head(hidden[:, :N_PATCH - 1].float())
    repeat_last = patches[:, :-1]                                   # baseline: next patch = current patch
    return F.mse_loss(pred, patches[:, 1:]).item(), F.mse_loss(repeat_last, patches[:, 1:]).item(), pred, patches

def probe_next_patch(steps=200, n_test=64):
    # Train a fresh head on frozen hidden states, then report its test MSE.
    probe = NextPatchHead(D).to(DEVICE)
    popt = torch.optim.AdamW(probe.parameters(), lr=1e-3)
    prng = np.random.default_rng(1)
    def states(xs):
        with torch.no_grad():
            patches = to_patches(np.stack(xs)).to(DEVICE)
            prompt = embed(torch.tensor(PROMPT_IDS, device=DEVICE)).expand(len(xs), -1, -1)
            h = lm.model(inputs_embeds=torch.cat([encoder(patches), prompt], dim=1)).last_hidden_state
        return h[:, :N_PATCH - 1].float(), patches[:, 1:]
    for _ in range(steps):
        h, target = states([make_series(prng)[0] for _ in range(BATCH)])
        loss = F.mse_loss(probe(h), target)
        popt.zero_grad(); loss.backward(); popt.step()
    test_rng = np.random.default_rng(99)
    h, target = states([make_series(test_rng)[0] for _ in range(n_test)])
    with torch.no_grad():
        return F.mse_loss(probe(h), target).item()

In [ ]:
if "vlm" in globals():                         # Section 6 ran: free its GPU memory
    del vlm; free_gpu()

from transformers import AutoModelForCausalLM, AutoTokenizer

LM_NAME = "Qwen/Qwen3-0.6B"

tok = AutoTokenizer.from_pretrained(LM_NAME)
lm = AutoModelForCausalLM.from_pretrained(LM_NAME, dtype=torch.float32).to(DEVICE)
n_lm = sum(p.numel() for p in lm.parameters())
lm.requires_grad_(False); lm.eval()
for layer in lm.model.layers:                     # LoRA on the query and value projections
    layer.self_attn.q_proj = LoRALinear(layer.self_attn.q_proj)
    layer.self_attn.v_proj = LoRALinear(layer.self_attn.v_proj)
lora_params = [p for n, p in lm.named_parameters() if n.endswith((".A", ".B"))]

embed = lm.get_input_embeddings()
D = lm.config.hidden_size
encoder, head = PatchEncoder(D).to(DEVICE), NextPatchHead(D).to(DEVICE)
PROMPT_IDS = tok("Describe the series:", add_special_tokens=False).input_ids
EOS = tok.eos_token_id
n_train = sum(p.numel() for p in [*encoder.parameters(), *head.parameters(), *lora_params])
print(f"Frozen LM: {n_lm / 1e6:.0f}M parameters | trainable (encoder, head, LoRA): {n_train / 1e6:.2f}M")

In [ ]:
LAMBDA, STEPS, BATCH = 1.0, 400, 16     # set LAMBDA = 0.0 to switch forecast grounding off
# bfloat16 autocast on GPUs that support it; plain float32 elsewhere (float16 can overflow in training).
use_amp = DEVICE == "cuda" and LLM_DTYPE == torch.bfloat16
opt = torch.optim.AdamW([{"params": [*encoder.parameters(), *head.parameters()], "lr": 1e-3},
                         {"params": lora_params, "lr": 3e-4}], weight_decay=0.01)
rng, history = np.random.default_rng(0), []
for step in range(STEPS + 1):
    xs, caps = zip(*[make_series(rng) for _ in range(BATCH)])
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_amp):
        ce, mse = losses(to_patches(np.stack(xs)).to(DEVICE), caps)
        loss = ce + LAMBDA * mse
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    history.append((ce.item(), mse.item()))
    if step % 50 == 0:
        print(f"step {step:3d} | caption CE {ce.item():.3f} | next-patch MSE {mse.item():.3f}")

In [ ]:
test_rng = np.random.default_rng(123)
test = [make_series(test_rng) for _ in range(60)]
hits_trend = hits_cycle = 0
for i, (x, cap) in enumerate(test):
    out = describe(x)
    hits_trend += cap.split()[0].lower() in out.lower()
    hits_cycle += re.search(rf"\b{cap.split()[-2]}\b", out) is not None
    if i < 5:
        print(f"truth: {cap:45s} model: {out}")
print(f"\ntrend accuracy {hits_trend / len(test):.0%} | cycle accuracy {hits_cycle / len(test):.0%}"
      f" (chance: 33% and 25%)")

mse, base, pred, patches = next_patch_mse([x for x, _ in test])
print(f"next-patch MSE {mse:.3f} vs {base:.3f} for 'repeat the current patch'")

fig, ax = plt.subplots(figsize=(12, 3))
truth = patches[0].flatten().cpu().numpy()
ax.plot(truth, color="0.3", label="series (standardised)")
ax.plot(np.arange(P, P * N_PATCH), pred[0].flatten().cpu().numpy(), color="tab:purple", label="next-patch prediction")
for k in range(1, N_PATCH):
    ax.axvline(k * P, color="0.85", lw=0.8)
ax.legend(fontsize=8); ax.set_title(test[0][1]); plt.tight_layout(); plt.show()

In [ ]:
print(f"probe next-patch MSE with LAMBDA = {LAMBDA}: {probe_next_patch():.3f}")

**Try this.** Set `LAMBDA = 0.0` and rerun the setup and training cells: the captions can still be learned, but nothing now asks the series tokens to keep track of what comes next. Fit the fresh probe head below under both settings — a lower probe error with `LAMBDA = 1.0` means the representation kept more temporal information. In Lotus the same idea runs at scale, with real pretraining corpora and a second phase on six downstream tasks.

## Wrap-up

* Numerical foundation models give zero-shot forecasts with uncertainty on domains they never saw — always compare them with naive and seasonal-naive baselines before trusting them.
* An instruction model can answer questions about a series written as text; the shuffle test tells you how much of that accuracy actually comes from the numbers.
* Forecast grounding asks the language-aligned representation to keep predicting what comes next — Section 7 shows the mechanics, Lotus the idea at scale.

**References.** Liu et al., *Time-MMD*, NeurIPS 2024 · Kong et al., *Time-MQA*, ACL 2025 · Merrill et al., Findings of EMNLP 2024 · Zhong et al., *Time-VLM*, ICML 2025 · Zhou & Yu, ICLR 2025 · Das et al., ICML 2024 · Ansari et al., *Chronos*, TMLR 2024 · Langer et al., *OpenTSLM*, 2026 · Do et al., *Lotus: Learning to Understand Time-series by Learning to Forecast*, arXiv 2610.18877, 2026.